# Gold Layer - Weather Impact Summary

## Purpose
Aggregate hourly weather observations to **daily summary by region** for operational risk assessment.

## Business Question
**"What weather conditions may affect operations by day and region?"**

## Output Table
`vattenfall_dev.gold.gold_weather_impact_summary`

**Grain:** `report_day` × `region`

**Measures:**
* Average wind speed
* Average temperature
* Total precipitation
* Weather risk score (0-9)
* Weather risk level (LOW/MODERATE/ELEVATED/HIGH/EXTREME)
* Alert counts (extreme conditions)

## Source
`vattenfall_dev.refined.silver_weather` (1,140 hourly records)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("✅ Setup complete")

In [0]:
# Load silver weather data
df_silver = spark.table("vattenfall_dev.refined.silver_weather")

print(f"✅ Loaded {df_silver.count():,} hourly weather records")
print(f"   Date range: {df_silver.agg(F.min('report_day')).first()[0]} to {df_silver.agg(F.max('report_day')).first()[0]}")
print(f"   Regions: {df_silver.select('region_normalized').distinct().count()}")

# Show sample
display(df_silver.select(
    'report_day', 'region_normalized', 'temperature_c', 
    'wind_speed_ms', 'precipitation_mm', 'is_extreme_temp', 'is_high_wind'
).limit(5))

In [0]:
# Aggregate to daily level by region
df_daily = df_silver.groupBy(
    "report_day",
    F.col("region_normalized").alias("region")
).agg(
    # Average temperature
    F.round(F.avg("temperature_c"), 1).alias("avg_temperature_c"),
    F.round(F.min("temperature_c"), 1).alias("min_temperature_c"),
    F.round(F.max("temperature_c"), 1).alias("max_temperature_c"),
    
    # Average wind speed
    F.round(F.avg("wind_speed_ms"), 1).alias("avg_wind_speed_ms"),
    F.round(F.max("wind_speed_ms"), 1).alias("max_wind_speed_ms"),
    
    # Total precipitation
    F.round(F.sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
    
    # Alert counts (extreme conditions)
    F.sum(F.when(F.col("is_extreme_temp"), 1).otherwise(0)).alias("extreme_temp_hours"),
    F.sum(F.when(F.col("is_high_wind"), 1).otherwise(0)).alias("high_wind_hours"),
    
    # Cloud cover
    F.round(F.avg("cloud_cover_percent"), 1).alias("avg_cloud_cover_percent"),
    
    # Hourly observations count
    F.count("*").alias("hourly_observations")
)

# Calculate weather risk scores (based on business logic)
df_gold = df_daily.withColumn(
    "temp_risk_score",
    F.when(F.col("avg_temperature_c") <= -4, 3)
     .when(F.col("avg_temperature_c") <= -2, 2)
     .when(F.col("avg_temperature_c") <= 0, 1)
     .otherwise(0)
).withColumn(
    "wind_risk_score",
    F.when(F.col("avg_wind_speed_ms") >= 12, 3)
     .when(F.col("avg_wind_speed_ms") >= 10, 2)
     .when(F.col("avg_wind_speed_ms") >= 8, 1)
     .otherwise(0)
).withColumn(
    "precip_risk_score",
    F.when(F.col("total_precipitation_mm") >= 5, 3)
     .when(F.col("total_precipitation_mm") >= 2, 2)
     .when(F.col("total_precipitation_mm") >= 0.5, 1)
     .otherwise(0)
).withColumn(
    "weather_risk_score",
    F.col("temp_risk_score") + F.col("wind_risk_score") + F.col("precip_risk_score")
).withColumn(
    "adjusted_risk_score",
    F.when(F.col("region") == "FI", F.col("weather_risk_score") * 1.2)
     .otherwise(F.col("weather_risk_score"))
).withColumn(
    "weather_risk_level",
    F.when(F.col("weather_risk_score") >= 7, "EXTREME")
     .when(F.col("weather_risk_score") >= 5, "HIGH")
     .when(F.col("weather_risk_score") >= 3, "ELEVATED")
     .when(F.col("weather_risk_score") >= 1, "MODERATE")
     .otherwise("LOW")
).orderBy("report_day", "region")

print(f"✅ Aggregated to {df_gold.count()} daily weather summaries")
print(f"   Grain: report_day × region")

# Show sample
display(df_gold.limit(10))

In [0]:
# Ensure gold schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS vattenfall_dev.gold")

# Write to Gold layer
table_name = "vattenfall_dev.gold.gold_weather_impact_summary"

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Successfully wrote to {table_name}")
print(f"   Records: {spark.table(table_name).count():,}")
print(f"   Columns: {len(spark.table(table_name).columns)}")

In [0]:
# Reload for validation
df_validate = spark.table("vattenfall_dev.gold.gold_weather_impact_summary")

print("=" * 70)
print("GOLD TABLE VALIDATION")
print("=" * 70)

# 1. Check grain uniqueness
print("\n1. GRAIN VALIDATION (report_day × region)")
print("-" * 70)
total_rows = df_validate.count()
unique_combos = df_validate.select("report_day", "region").distinct().count()
print(f"   Total rows: {total_rows}")
print(f"   Unique day-region combinations: {unique_combos}")
print(f"   ✅ Grain is unique: {total_rows == unique_combos}")

# 2. Check for nulls in key columns
print("\n2. NULL CHECK")
print("-" * 70)
for col in ["report_day", "region", "avg_temperature_c", "avg_wind_speed_ms"]:
    null_count = df_validate.filter(F.col(col).isNull()).count()
    status = "✅ PASS" if null_count == 0 else f"❌ {null_count} nulls"
    print(f"   {col:.<30} {status}")

# 3. Date coverage
print("\n3. DATE COVERAGE")
print("-" * 70)
date_stats = df_validate.agg(
    F.min("report_day").alias("start_date"),
    F.max("report_day").alias("end_date"),
    F.countDistinct("report_day").alias("unique_days")
).first()
print(f"   Start date: {date_stats['start_date']}")
print(f"   End date: {date_stats['end_date']}")
print(f"   Unique days: {date_stats['unique_days']}")

# 4. Regional weather summary
print("\n4. REGIONAL WEATHER SUMMARY")
print("-" * 70)
df_validate.groupBy("region").agg(
    F.count("*").alias("days"),
    F.round(F.avg("avg_temperature_c"), 1).alias("avg_temp"),
    F.round(F.avg("avg_wind_speed_ms"), 1).alias("avg_wind"),
    F.round(F.avg("weather_risk_score"), 1).alias("avg_risk")
).orderBy("region").show()

# 5. Weather risk level distribution
print("\n5. WEATHER RISK LEVEL DISTRIBUTION")
print("-" * 70)
df_validate.groupBy("weather_risk_level").agg(
    F.count("*").alias("day_count")
).orderBy("weather_risk_level").show()

# 6. Top 5 highest risk days
print("\n6. TOP 5 HIGHEST WEATHER RISK DAYS")
print("-" * 70)
df_validate.orderBy(F.col("weather_risk_score").desc()).limit(5).select(
    "report_day", "region", "weather_risk_level", "weather_risk_score",
    "avg_temperature_c", "avg_wind_speed_ms", "total_precipitation_mm"
).show(truncate=False)

print("\n" + "=" * 70)
print("✅ Gold table validation complete")
print("=" * 70)

## ✅ Business Question Answered

### **"What weather conditions may affect operations by day and region?"**

---

## Key Findings (January 1-15, 2024)

### 🚨 **CRITICAL: Entire Period Was High-Risk**

**100% of days experienced HIGH or EXTREME weather risk:**
* **26 days (43%)** - EXTREME risk
* **34 days (57%)** - HIGH risk
* **0 days** - Moderate, Low, or calm conditions

**Interpretation:** Operations were under **continuous weather stress** throughout the 2-week period.

---

### 🌡️ **Temperature Conditions**

**All regions experienced sub-zero temperatures:**
* Finland (FI): **-2.8°C average** (coldest)
* Denmark (DK): **-2.6°C average**
* Sweden (SE): **-2.6°C average**
* Norway (NO): **-2.4°C average** (warmest, but still freezing)

**Operational Impact:**
* ❄️ Equipment freeze risk (mechanical failures, sensor issues)
* ⚡ Increased heating demand (grid load spikes)
* 🔧 Material brittleness (infrastructure stress)

**Coldest Days:**
* Jan 15 FI: **-4.3°C** (extreme)
* Jan 1 FI: **-4.1°C**
* Jan 6 DK: **-4.1°C**

---

### 💨 **Wind Conditions**

**Sustained high winds across all regions:**
* Norway (NO): **10.1 m/s average** (22.6 mph)
* Sweden (SE): **10.1 m/s average**
* Finland (FI): **9.9 m/s average**
* Denmark (DK): **9.8 m/s average**

**Operational Impact:**
* ⚠️ **Wind is the PRIMARY risk factor** - all incident days had wind ≥ 8 m/s
* 🌲 Tree fall risk on power lines
* 🔌 Physical stress on overhead infrastructure
* 🏗️ Construction/maintenance work restrictions

**Highest Wind Days:**
* Jan 13 FI: **12.9 m/s** (28.9 mph) - severe
* Jan 3 NO: **12.1 m/s** (27.1 mph)
* Jan 15 FI: **11.9 m/s** (26.6 mph)

---

### 🌧️ **Precipitation Conditions**

**Heavy precipitation events (winter rain/snow mix):**

**Top 5 Wettest Days:**
1. Jan 13 FI: **68.0 mm** (extreme)
2. Jan 15 FI: **56.9 mm** (extreme)
3. Jan 1 FI: **52.7 mm** (extreme)
4. Jan 6 DK: **41.4 mm** (very high)
5. Jan 3 NO: **29.9 mm** (high)

**Operational Impact:**
* ⚡ Ice accumulation on lines (weight stress, flashover risk)
* 🌊 Flooding risk for substations
* 🚗 Access difficulties for maintenance crews

**Finland Pattern:** 3 of top 5 wettest days occurred in Finland

---

### 🎯 **Regional Operational Risk Summary**

#### **All Regions: Similar Baseline Risk**

| Region | Avg Risk Score | Risk Profile |
|--------|----------------|-------------|
| **Denmark (DK)** | 6.5 | High wind + moderate cold + heavy precip |
| **Sweden (SE)** | 6.5 | High wind + moderate cold + heavy precip |
| **Finland (FI)** | 6.3 | **Most weather-sensitive** (1.2× multiplier) |
| **Norway (NO)** | 6.3 | High wind + lightest precip |

**Note:** Despite similar risk scores, **Finland gets priority attention** due to historical weather-related incident concentration (7 of 10 events in validation data).

---

### ⚠️ **Top 5 Highest Risk Days (Operations Should Have Been on Alert)**

#### **1. Jan 6, 2024 - Denmark**
* **Risk:** EXTREME (score 8)
* **Conditions:** -4.1°C + 10.3 m/s wind + 41.4 mm precip
* **Triple Threat:** ✅ All three factors at risk levels

#### **2. Jan 3, 2024 - Norway**
* **Risk:** EXTREME (score 8)
* **Conditions:** -2.4°C + **12.1 m/s wind** + 29.9 mm precip
* **Key Factor:** Highest wind speed of the period

#### **3. Jan 1, 2024 - Finland**
* **Risk:** EXTREME (score 8)
* **Conditions:** -4.1°C + 10.5 m/s wind + 52.7 mm precip
* **Actual Outcome:** 1 critical event in Turku (2,202 customers affected)

#### **4. Jan 13, 2024 - Finland**
* **Risk:** EXTREME (score 8)
* **Conditions:** -2.5°C + **12.9 m/s wind** + **68.0 mm precip**
* **Key Factor:** Highest precipitation AND wind of entire period

#### **5. Jan 15, 2024 - Finland**
* **Risk:** EXTREME (score 8)
* **Conditions:** **-4.3°C** + 11.9 m/s wind + 56.9 mm precip
* **Key Factor:** Coldest day of the period

---

## 📊 **Operational Recommendations**

### **Immediate Actions During Similar Conditions:**

1. **Pre-Position Crews**
   * When weather_risk_score ≥ 7: Deploy emergency crews to regions
   * Focus on Finland during EXTREME conditions (historical vulnerability)

2. **Asset Monitoring**
   * Increase inspection frequency for aging infrastructure
   * Monitor substations in areas with wind ≥ 10 m/s

3. **Customer Communication**
   * Issue proactive advisories when risk level = HIGH or EXTREME
   * Set expectations for potential service interruptions

4. **Maintenance Restrictions**
   * Halt non-critical maintenance when wind ≥ 12 m/s
   * Avoid overhead work during freezing rain events

5. **Load Management**
   * Anticipate demand spikes during extreme cold periods
   * Coordinate with market operations for price volatility

### **Strategic Infrastructure Investment:**

* **Finland Priority:** Upgrade wind resistance on substations (most weather-sensitive)
* **Cold-Weather Hardening:** Equipment rated for -10°C to -20°C conditions
* **Ice Prevention:** Install line heaters or vibration systems on critical spans

---

## 🎯 **Answer Summary**

**"What weather conditions may affect operations by day and region?"**

**EVERY DAY in January 1-15, 2024 posed operational risk due to:**
1. **Sustained high winds (8-13 m/s)** - PRIMARY risk factor
2. **Sub-zero temperatures (-4.3°C to -2.4°C)** - Equipment stress
3. **Heavy precipitation (0-68 mm daily)** - Ice accumulation

**43% of days were EXTREME risk, 57% were HIGH risk.**

**No region was spared** - all experienced similar conditions, but **Finland requires priority attention** due to historical incident concentration.

**This table enables proactive crew deployment, maintenance planning, and customer communication** based on weather forecasts.